### Persistence in LangGraph


In [12]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [13]:
load_dotenv()  # Load environment variables from .env file

True

In [14]:
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [15]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [16]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [17]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [18]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [19]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job?\n\nBecause it needed to make some **dough**!',
 'explanation': 'This joke is a classic example of a **pun**!\n\nThe humor comes from the word **"dough,"** which has two distinct meanings relevant to the joke:\n\n1.  **Literal meaning (for pizza):** "Dough" is the unbaked mixture of flour, water, yeast, etc., that forms the base of a pizza crust. Pizza *is made from* dough, essentially.\n2.  **Slang meaning (for people):** In informal language, "dough" is a common slang term for **money**. (e.g., "I need to earn some dough," "He\'s got a lot of dough.")\n\n**How the joke works:**\n\n*   The setup "Why did the pizza get a job?" personifies the pizza, treating it like a human who needs to earn a living.\n*   The punchline "Because it needed to make some **dough**!" plays on both meanings simultaneously:\n    *   Like a person, the pizza needs to "make some dough" (earn money) to survive.\n    *   But for a pizza, "making dough" als

In [21]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it needed to make some **dough**!', 'explanation': 'This joke is a classic example of a **pun**!\n\nThe humor comes from the word **"dough,"** which has two distinct meanings relevant to the joke:\n\n1.  **Literal meaning (for pizza):** "Dough" is the unbaked mixture of flour, water, yeast, etc., that forms the base of a pizza crust. Pizza *is made from* dough, essentially.\n2.  **Slang meaning (for people):** In informal language, "dough" is a common slang term for **money**. (e.g., "I need to earn some dough," "He\'s got a lot of dough.")\n\n**How the joke works:**\n\n*   The setup "Why did the pizza get a job?" personifies the pizza, treating it like a human who needs to earn a living.\n*   The punchline "Because it needed to make some **dough**!" plays on both meanings simultaneously:\n    *   Like a person, the pizza needs to "make some dough" (earn money) to survive.\n    *   But for a pizza,

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it needed to make some **dough**!', 'explanation': 'This joke is a classic example of a **pun**!\n\nThe humor comes from the word **"dough,"** which has two distinct meanings relevant to the joke:\n\n1.  **Literal meaning (for pizza):** "Dough" is the unbaked mixture of flour, water, yeast, etc., that forms the base of a pizza crust. Pizza *is made from* dough, essentially.\n2.  **Slang meaning (for people):** In informal language, "dough" is a common slang term for **money**. (e.g., "I need to earn some dough," "He\'s got a lot of dough.")\n\n**How the joke works:**\n\n*   The setup "Why did the pizza get a job?" personifies the pizza, treating it like a human who needs to earn a living.\n*   The punchline "Because it needed to make some **dough**!" plays on both meanings simultaneously:\n    *   Like a person, the pizza needs to "make some dough" (earn money) to survive.\n    *   But for a pizza